# Mt. Hood Ash Dispersal — 3-POI Sweep Analysis

Post-processing, visualization, and export for the Tephra2 parameter sweep produced by `tephra2_sweep.ipynb`,
evaluated at three points of interest (POI) -- Rhododendron, Parkdale, and Government Camp -- and written as
`rhododendron.csv`, `parkdale.csv`, and `govt_camp.csv`.

This is point-based analysis, not a spatial/regional one: the sweep no longer evaluates a grid across the
region, only these 3 fixed locations, so there is no hazard map, exceedance-probability map, or GeoTIFF export
here -- see `prelim_tephra2_working.ipynb` for a spatial isomass map of one specific scenario.

### Hazard thresholds

Ash loading thresholds below are from Wilson et al. (2014) and Jenkins et al. (2015):

| Threshold (kg/m²) | Impact |
|---:|---|
| 1 | Transport and agriculture disruption |
| 10 | Crop damage, infrastructure disruption |
| 100 | Roof collapse risk |
| 1,000 | Severe structural damage |

These thresholds are load-based, not volcano-type specific, and assume a bulk ash density of ~1,000 kg/m³.
Mt. Hood's fine lithic ash may have a lower bulk density (~500–800 kg/m³), so a given kg/m² loading corresponds
to a *thicker* deposit here than at a volcano producing denser ash — treat the mapped kg/m² values as the
physically robust quantity, and convert to a deposit thickness only with an explicit density assumption in mind.

### References
Wilson, T.M. et al. (2014); Jenkins, S.F. et al. (2015); Scott et al. (2025); Buckland et al. (2022);
Mannen (2020); Biass et al. (2016); Pardini et al. (2016); Kawamoto & Ui (2021).

## Section 1 — Setup & Data Loading

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


In [ ]:
HAZARD_THRESHOLDS = [1, 10, 100, 1000]  # kg/m^2 -- Wilson et al. (2014), Jenkins et al. (2015)

# Points of interest (must match tephra2_sweep.ipynb)
poi_names = ["Rhododendron", "Parkdale", "Govt. Camp"]
poi_locations = {
    "Rhododendron": (45.329563, -121.911191),
    "Parkdale": (45.519839, -121.596742),
    "Govt. Camp": (45.30351444525525, -121.75331979925848),
}
poi_files = {
    name: f"{name.lower().replace('.', '').replace(' ', '_')}.csv"
    for name in poi_names
}


In [ ]:
poi_data = {}
for name, fname in poi_files.items():
    df = pd.read_csv(fname)
    # Tephra2 can underflow to values below double precision's usable range; treat those as exact zero
    df.loc[df['mass_kg_m2'] < 1e-300, 'mass_kg_m2'] = 0.0
    poi_data[name] = df
    print(f"{name}: {len(df):,} runs loaded from {fname}")


In [ ]:
def cell_edges(centers):
    """Quadrilateral edges for pcolormesh(shading='flat') that stay pinned to the data range.

    pcolormesh(shading='auto') treats the given coordinates as cell centers and extrapolates the
    outer edges by half the center-to-center spacing -- fine with many finely-spaced sweep steps,
    but with only a few steps that extrapolation is huge and pushes the axis well past the actual
    swept range. Clamping the outermost edges to the outermost centers keeps the axis exactly at
    [min(centers), max(centers)] regardless of spacing.
    """
    centers = np.asarray(centers, dtype=float)
    if len(centers) == 1:
        return np.array([centers[0] - 0.5, centers[0] + 0.5])
    midpoints = (centers[:-1] + centers[1:]) / 2
    return np.concatenate(([centers[0]], midpoints, [centers[-1]]))


In [ ]:
def poi_summary(name):
    """Summary stats for one POI: max/mean/median thickness, hazard exceedance probabilities,
    and the parameter combination that produced the maximum."""
    df = poi_data[name]
    max_row = df.loc[df['mass_kg_m2'].idxmax()]

    summary = {
        'location': name,
        'n_runs': len(df),
        'max_mass_kg_m2': float(df['mass_kg_m2'].max()),
        'mean_mass_kg_m2': float(df['mass_kg_m2'].mean()),
        'median_mass_kg_m2': float(df['mass_kg_m2'].median()),
        'max_scenario_plume_height_m': int(max_row['plume_height']),
        'max_scenario_eruption_mass_kg': float(max_row['eruption_mass']),
        'max_scenario_diffusion_coef_m2_s': float(max_row['diffusion_coef']),
    }
    for threshold in HAZARD_THRESHOLDS:
        summary[f'prob_exceed_{threshold}_kg_m2'] = float((df['mass_kg_m2'] >= threshold).mean())
    return summary


def hazard_level_label(max_mass_kg_m2):
    if max_mass_kg_m2 >= 1000:
        return "Severe structural damage risk (>=1000 kg/m^2)"
    elif max_mass_kg_m2 >= 100:
        return "Roof collapse risk (>=100 kg/m^2)"
    elif max_mass_kg_m2 >= 10:
        return "Crop damage / infrastructure disruption (>=10 kg/m^2)"
    elif max_mass_kg_m2 >= 1:
        return "Transport / agriculture disruption (>=1 kg/m^2)"
    else:
        return "Below mapped hazard thresholds (<1 kg/m^2)"


## Section 2 — Configuration

Toggle which analyses run below.

In [ ]:
run_plume_envelope_plots = True
run_heatmaps              = True
run_summary_stats         = True
run_export_geojson        = True


## Section 3 — Plume Height vs. Ash Thickness Envelopes

For each POI, at every swept plume height, this plots the min/max ash thickness envelope across all
combinations of eruption mass and diffusion coefficient. This shows how much a location's exposure is
driven by plume height alone versus the other two parameters.

In [ ]:
def plot_envelope(name):
    df = poi_data[name]
    envelope = df.groupby('plume_height')['mass_kg_m2'].agg(['min', 'max', 'median'])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.fill_between(envelope.index, envelope['min'] + 1e-6, envelope['max'],
                     color='tab:orange', alpha=0.3, label='min-max envelope')
    ax.plot(envelope.index, envelope['median'], color='tab:orange', lw=1.5, label='median')

    for threshold in HAZARD_THRESHOLDS:
        ax.axhline(threshold, color='gray', ls=':', lw=0.8)
        ax.text(envelope.index.max(), threshold, f' {threshold} kg/m²', va='bottom', ha='right',
                fontsize=7, color='gray')

    ax.set_yscale('log')
    ax.set_xlabel('Plume height (m asl)')
    ax.set_ylabel('Ash thickness (kg/m²)')
    ax.set_title(f'{name} — ash thickness vs. plume height\n(envelope across eruption mass & diffusion coefficient)')
    ax.legend()
    fig.tight_layout()

    slug = name.lower().replace('.', '').replace(' ', '_')
    fig.savefig(f'envelope_{slug}.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if run_plume_envelope_plots:
    plot_envelope("Rhododendron")


In [ ]:
if run_plume_envelope_plots:
    plot_envelope("Parkdale")


In [ ]:
if run_plume_envelope_plots:
    plot_envelope("Govt. Camp")


## Section 4 — Plume Height vs. Eruption Mass Heatmaps

Ash thickness as a function of plume height (x) and eruption mass (y), averaged over diffusion coefficient, at
each POI location.

In [ ]:
if run_heatmaps:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, name in zip(axes, poi_names):
        df = poi_data[name]
        pivot = df.pivot_table(index='eruption_mass', columns='plume_height',
                                values='mass_kg_m2', aggfunc='mean')

        positive_values = pivot.values[pivot.values > 0]
        norm = LogNorm(vmin=max(positive_values.min(), 1e-3), vmax=pivot.values.max()) if len(positive_values) else None
        x_edges = cell_edges(pivot.columns.values)
        y_edges = cell_edges(pivot.index.values)
        pcm = ax.pcolormesh(x_edges, y_edges, pivot.values,
                             norm=norm, cmap='YlOrRd', shading='flat')
        fig.colorbar(pcm, ax=ax, label='Ash thickness (kg/m²)')
        ax.set_xlabel('Plume height (m asl)')
        ax.set_ylabel('Eruption mass (kg)')
        ax.set_yscale('log')
        ax.set_title(name)

    fig.suptitle('Ash thickness vs. plume height & eruption mass (diffusion coefficient averaged)', y=1.03)
    fig.tight_layout()
    fig.savefig('plume_height_eruption_mass_heatmaps.png', dpi=150, bbox_inches='tight')
    plt.show()


## Section 5 — Summary Statistics & Hazard Exceedance

For each POI: maximum/mean/median ash thickness across all sweep runs, the parameter combination that produced
the maximum, and the probability (fraction of sweep runs) that meets or exceeds each hazard threshold.

In [ ]:
if run_summary_stats:
    summary_df = pd.DataFrame([poi_summary(name) for name in poi_names]).set_index('location')

    for name in poi_names:
        s = summary_df.loc[name]
        print(f"{name}:")
        print(f"  max ash thickness   = {s['max_mass_kg_m2']:.3g} kg/m^2")
        print(f"  mean / median       = {s['mean_mass_kg_m2']:.3g} / {s['median_mass_kg_m2']:.3g} kg/m^2")
        print(f"  worst-case scenario = plume_height={s['max_scenario_plume_height_m']:.0f} m, "
              f"eruption_mass={s['max_scenario_eruption_mass_kg']:.3g} kg, "
              f"diffusion_coef={s['max_scenario_diffusion_coef_m2_s']:.3g} m^2/s")
        for threshold in HAZARD_THRESHOLDS:
            p = s[f'prob_exceed_{threshold}_kg_m2']
            print(f"  P(loading >= {threshold:>4} kg/m^2) = {p:.1%}")
        print()

    summary_df


## Section 6 — Export for QGIS

Exports the 3 POIs as a **point layer** (GeoJSON, WGS84 / EPSG:4326), not a raster -- with only 3 fixed
locations there's no spatial surface to interpolate, so a "map" built from them would misrepresent the hazard
rather than clarify it. Each point instead carries summary attributes (max/mean/median thickness, hazard
exceedance probabilities, worst-case scenario, and a plain-language hazard level) so the layer is directly
readable in QGIS by someone without domain background -- click a point, read what it says.

In [ ]:
def build_poi_geojson():
    features = []
    for name in poi_names:
        lat, lon = poi_locations[name]
        summary = poi_summary(name)
        summary['hazard_level'] = hazard_level_label(summary['max_mass_kg_m2'])
        summary.pop('location')  # redundant with the feature's "name" property

        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lon, lat]},
            "properties": {"name": name, **summary},
        })

    return {"type": "FeatureCollection", "features": features}


In [ ]:
if run_export_geojson:
    geojson = build_poi_geojson()
    with open("poi_hazard_summary.geojson", "w") as f:
        json.dump(geojson, f, indent=2)
    print("Wrote poi_hazard_summary.geojson")
    geojson
